# 07 — Comparação de modelos e thresholds

Compara Regressão Logística balanceada, Random Forest e XGBoost e seleciona thresholds pela maximização do F2-score.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier
import joblib

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_MODELOS = os.path.join(PASTA_PROJETO, "outputs", "modelos")
os.makedirs(PASTA_MODELOS, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))
df["br_categoria"] = df["br"].astype("Int64").astype("string").fillna("Não informado")

In [ ]:
colunas_tracado = [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]
categoricas = ["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas = ["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"] + colunas_tracado
features = categoricas + numericas
TARGET = "acidente_grave"

treino = df[df["ano"] <= 2023].copy()
validacao = df[df["ano"] == 2024].copy()
teste_final = df[df["ano"] == 2025].copy()

X_train, y_train = treino[features], treino[TARGET]
X_val, y_val = validacao[features], validacao[TARGET]

neg = (y_train==0).sum(); pos = (y_train==1).sum()
scale_pos_weight = neg/pos
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
def enc():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse=True)

def pre():
    return ColumnTransformer([
        ("categoricas", enc(), categoricas),
        ("numericas", StandardScaler(), numericas)
    ])

def avaliar(modelo, X, y, nome, threshold=.5):
    prob = modelo.predict_proba(X)[:,1]
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y,pred).ravel()
    return {
        "modelo": nome, "threshold": threshold,
        "accuracy": accuracy_score(y,pred),
        "precision": precision_score(y,pred,zero_division=0),
        "recall": recall_score(y,pred,zero_division=0),
        "f1": f1_score(y,pred,zero_division=0),
        "f2": fbeta_score(y,pred,beta=2,zero_division=0),
        "roc_auc": roc_auc_score(y,prob),
        "pr_auc": average_precision_score(y,prob),
        "verdadeiro_negativo": tn, "falso_positivo": fp,
        "falso_negativo": fn, "verdadeiro_positivo": tp
    }

In [ ]:
modelos = {
    "Logística Balanceada": Pipeline([
        ("preprocessamento", pre()),
        ("modelo", LogisticRegression(solver="saga", class_weight="balanced", max_iter=700, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessamento", pre()),
        ("modelo", RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=10,
                                           class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("preprocessamento", pre()),
        ("modelo", XGBClassifier(n_estimators=500, learning_rate=.05, max_depth=6, min_child_weight=5,
                                 subsample=.8, colsample_bytree=.8, scale_pos_weight=scale_pos_weight,
                                 objective="binary:logistic", eval_metric="logloss", n_jobs=-1, random_state=42))
    ])
}

metricas = []
for nome, modelo in modelos.items():
    print("Treinando", nome)
    modelo.fit(X_train, y_train)
    metricas.append(avaliar(modelo, X_val, y_val, nome))
    joblib.dump(modelo, os.path.join(PASTA_MODELOS, nome.lower().replace(" ","_") + ".joblib"))

comparacao = pd.DataFrame(metricas).sort_values("f2", ascending=False)
display(comparacao)
comparacao.to_csv(os.path.join(PASTA_MODELOS, "comparacao_modelos_balanceados.csv"), index=False, encoding="utf-8-sig")

In [ ]:
def analisar_thresholds(modelo, X, y, nome):
    prob = modelo.predict_proba(X)[:,1]
    rows = []
    for t in np.arange(.10, .81, .02):
        pred = (prob >= t).astype(int)
        rows.append({
            "modelo": nome, "threshold": t,
            "precision": precision_score(y,pred,zero_division=0),
            "recall": recall_score(y,pred,zero_division=0),
            "f1": f1_score(y,pred,zero_division=0),
            "f2": fbeta_score(y,pred,beta=2,zero_division=0)
        })
    return pd.DataFrame(rows)

thresholds = pd.concat([
    analisar_thresholds(modelos[n], X_val, y_val, n) for n in modelos
], ignore_index=True)

melhores = (thresholds.sort_values(["modelo","f2"], ascending=[True,False])
            .groupby("modelo", as_index=False).first())

display(melhores)
thresholds.to_csv(os.path.join(PASTA_MODELOS, "analise_thresholds.csv"), index=False, encoding="utf-8-sig")
melhores.to_csv(os.path.join(PASTA_MODELOS, "melhores_thresholds.csv"), index=False, encoding="utf-8-sig")
print("2025 permanece reservado.")